In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import requests
import pandas as pd
import numpy as np
from tqdm import tqdm
import time

In [ ]:
PAIRS = [
    "BTC-USDT",
    "ETH-USDT",
    "SOL-USDT"
]

TIMEFRAMES = [
    "30m",
    "1H"
]

MAX_CANDLES = 100

SAVE_PATH = "/content/drive/MyDrive/AI_MANUAL_PROJECT/raw_data/"

In [ ]:
def download_history(symbol, timeframe):

    all_data = []

    after = ""

    while True:

        params = {
            "instId": symbol,
            "bar": timeframe,
            "limit": MAX_CANDLES
        }

        if after != "":
            params["after"] = after

        url = "https://www.okx.com/api/v5/market/history-candles"

        response = requests.get(
            url,
            params=params
        )

        data = response.json()

        if "data" not in data:
            break

        candles = data["data"]

        if len(candles) == 0:
            break

        all_data.extend(candles)

        after = candles[-1][0]

        print(
            symbol,
            timeframe,
            len(all_data)
        )

        time.sleep(0.15)

        if len(all_data) > 120000:
            break

    df = pd.DataFrame(
        all_data,
        columns=[
            "time",
            "open",
            "high",
            "low",
            "close",
            "vol",
            "volCcy",
            "volCcyQuote",
            "confirm"
        ]
    )

    numeric_cols = [
        "open",
        "high",
        "low",
        "close",
        "vol"
    ]

    df[numeric_cols] = (
        df[numeric_cols]
        .astype(float)
    )

    df["time"] = pd.to_datetime(
        df["time"].astype(float),
        unit="ms"
    )

    df = (
        df
        .sort_values("time")
        .reset_index(drop=True)
    )

    return df

In [ ]:
for pair in PAIRS:

    for tf in TIMEFRAMES:

        print(
            "\nDownloading",
            pair,
            tf
        )

        df = download_history(
            pair,
            tf
        )

        filename = (
            pair.replace("-","_")
            +
            "_"
            +
            tf
            +
            ".csv"
        )

        save_file = SAVE_PATH + filename

        df.to_csv(
            save_file,
            index=False
        )

        print(
            "Saved:",
            filename
        )

print("RAW DATA COMPLETE")

Streaming output truncated to the last 5000 lines.
BTC-USDT 30m 40600
BTC-USDT 30m 40700
BTC-USDT 30m 40800
BTC-USDT 30m 40900
BTC-USDT 30m 41000
BTC-USDT 30m 41100
BTC-USDT 30m 41200
BTC-USDT 30m 41300
BTC-USDT 30m 41400
BTC-USDT 30m 41500
BTC-USDT 30m 41600
BTC-USDT 30m 41700
BTC-USDT 30m 41800
BTC-USDT 30m 41900
BTC-USDT 30m 42000
BTC-USDT 30m 42100
BTC-USDT 30m 42200
BTC-USDT 30m 42300
BTC-USDT 30m 42400
BTC-USDT 30m 42500
BTC-USDT 30m 42600
BTC-USDT 30m 42700
BTC-USDT 30m 42800
BTC-USDT 30m 42900
BTC-USDT 30m 43000
BTC-USDT 30m 43100
BTC-USDT 30m 43200
BTC-USDT 30m 43300
BTC-USDT 30m 43400
BTC-USDT 30m 43500
BTC-USDT 30m 43600
BTC-USDT 30m 43700
BTC-USDT 30m 43800
BTC-USDT 30m 43900
BTC-USDT 30m 44000
BTC-USDT 30m 44100
BTC-USDT 30m 44200
BTC-USDT 30m 44300
BTC-USDT 30m 44400
BTC-USDT 30m 44500
BTC-USDT 30m 44600
BTC-USDT 30m 44700
BTC-USDT 30m 44800
BTC-USDT 30m 44900
BTC-USDT 30m 45000
BTC-USDT 30m 45100
BTC-USDT 30m 45200
BTC-USDT 30m 45300
BTC-USDT 30m 45400
BTC-USDT 30m 45500

In [ ]:
def load_raw(pair, timeframe):

    filename = (
        "/content/drive/MyDrive/AI_MANUAL_PROJECT/raw_data/"
        +
        pair.replace("-", "_")
        +
        "_"
        +
        timeframe
        +
        ".csv"
    )

    df = pd.read_csv(filename)

    df["time"] = pd.to_datetime(df["time"])

    return df

In [ ]:
def add_model_features(df):

    # EMA20
    df["EMA20"] = (
        df["close"]
        .ewm(span=20, adjust=False)
        .mean()
    )

    # EMA50
    df["EMA50"] = (
        df["close"]
        .ewm(span=50, adjust=False)
        .mean()
    )

    # RSI
    delta = df["close"].diff()

    gain = delta.clip(lower=0)

    loss = -delta.clip(upper=0)

    avg_gain = gain.rolling(14).mean()

    avg_loss = loss.rolling(14).mean()

    rs = avg_gain/(avg_loss+1e-8)

    df["RSI"] = 100-(100/(1+rs))

    # ATR
    tr1 = df["high"]-df["low"]

    tr2 = abs(df["high"]-df["close"].shift())

    tr3 = abs(df["low"]-df["close"].shift())

    tr = pd.concat(
        [tr1,tr2,tr3],
        axis=1
    ).max(axis=1)

    df["ATR"] = tr.rolling(14).mean()

    df["NATR"] = (
        df["ATR"]
        /
        df["close"]
    )*100

    # BB WIDTH
    middle = df["close"].rolling(20).mean()

    std = df["close"].rolling(20).std()

    upper = middle+2*std

    lower = middle-2*std

    df["BB_WIDTH"] = upper-lower

    # CHAIKIN VOL
    ema_range = (
        (df["high"]-df["low"])
        .ewm(span=10)
        .mean()
    )

    df["CHAIKIN_VOL"] = (
        ema_range
        .pct_change(10)
    )*100

    # VQI
    df["VQI"] = (
        abs(
            df["close"]-
            df["open"]
        )
        /
        (
            df["high"]
            -
            df["low"]
            +
            1e-5
        )
    )

    # Trend strength
    sma10 = df["close"].rolling(10).mean()

    sma30 = df["close"].rolling(30).mean()

    df["TREND_STRENGTH"] = (
        abs(sma10-sma30)
        /
        (sma30+1e-8)
    )*100

    # Channel position
    upper20 = (
        df["high"]
        .rolling(20)
        .max()
    )

    lower20 = (
        df["low"]
        .rolling(20)
        .min()
    )

    df["CHANNEL_POSITION"] = (
        (
            df["close"]
            -
            lower20
        )
        /
        (
            upper20
            -
            lower20
            +
            1e-8
        )
    )

    return df

In [ ]:
def slope_signal(df):

    y = (
        df["high"]
        .rolling(4)
        .mean()
        -
        df["low"]
        .rolling(4)
        .mean()
    )

    x = abs(
        df["close"]
        -
        df["open"]
    )

    df["SLOPE_SIGNAL"] = (
        y
        /
        (x+1e-8)
    )

    return df

In [ ]:
def power_score(df):

    body = abs(
        df["close"]
        -
        df["open"]
    )

    rng = (
        df["high"]
        -
        df["low"]
    )

    volume_factor = (
        df["vol"]
        /
        (
            df["vol"]
            .rolling(20)
            .mean()
            +
            1e-8
        )
    )

    score = (
        body
        /
        (rng+1e-8)
    )*100

    df["POWER_SCORE"] = (
        score*0.7
        +
        volume_factor*30
    )

    return df

In [ ]:
def active_passive(df):

    y = (
        df["high"]
        -
        df["low"]
    )

    x = abs(
        df["close"]
        -
        df["open"]
    )

    k = 1.5

    cond = y > (k*x)

    df["ACTIVE_PASSIVE"] = np.where(
        cond,
        "ACTIVE",
        "PASSIVE"
    )

    return df

In [ ]:
def market_state(df):

    df["MARKET_STATE"]="NORMAL"

    df.loc[
        df["TREND_STRENGTH"]>0.2,
        "MARKET_STATE"
    ]="TRENDING"

    df.loc[
        df["BB_WIDTH"]<0.5,
        "MARKET_STATE"
    ]="RANGING"

    return df

In [ ]:
def financial_strength(df):

    strength = (
        df["POWER_SCORE"]
        *
        df["TREND_STRENGTH"]
    )

    df["FINANCIAL_STRENGTH"] = strength

    return df

In [ ]:
def day_night(df):

    hour = pd.to_datetime(
        df["time"]
    ).dt.hour

    df["DAY_NIGHT"] = np.where(
        (hour>=8)&(hour<=20),
        "DAY",
        "NIGHT"
    )

    return df

In [ ]:
def frequency_type(df, timeframe):

    if timeframe == "30m":
        df["FREQUENCY_TYPE"] = "MID_30M"

    elif timeframe == "1H":
        df["FREQUENCY_TYPE"] = "MID_1H"

    return df

In [ ]:
def candle_pattern(df):

    body = abs(
        df["close"]
        -
        df["open"]
    )

    rng = (
        df["high"]
        -
        df["low"]
    )

    df["CANDLE_PATTERN"]="NORMAL"

    df.loc[
        body<0.1*rng,
        "CANDLE_PATTERN"
    ]="DOJI"

    df.loc[
        body>0.8*rng,
        "CANDLE_PATTERN"
    ]="STRONG"

    return df

In [ ]:
def future_return(df):

    future_close = (
        df["close"]
        .shift(-4)
    )

    df["FUTURE_RETURN"] = (
        (
            future_close
            -
            df["close"]
        )
        /
        df["close"]
    ) * 100

    return df

In [ ]:
def target(df):

    df["TARGET"] = (

        abs(
            df["FUTURE_RETURN"]
        ) > 0.85

    ).astype(int)

    return df

In [ ]:
def interesting_signal(df):

    cond = (

        (df["POWER_SCORE"] > 60)

        &

        (df["TREND_STRENGTH"] > 0.15)

        &

        (df["SLOPE_SIGNAL"] > 1)

    )

    df["INTERESTING_SIGNAL"] = (
        cond.astype(int)
    )

    return df

In [ ]:
dataset = []

for pair in PAIRS:

    for tf in TIMEFRAMES:

        print(pair, tf)

        df = load_raw(
            pair,
            tf
        )

        df = add_model_features(df)

        df = slope_signal(df)

        df = power_score(df)

        df = active_passive(df)

        df = market_state(df)

        df = financial_strength(df)

        df = candle_pattern(df)

        df = frequency_type(
            df,
            tf
        )

        df = future_return(df)

        df = target(df)

        df = interesting_signal(df)

        df["pair"] = pair

        df["timeframe"] = tf

        df = df[
            df["INTERESTING_SIGNAL"] == 1
        ]

        dataset.append(df)

BTC-USDT 30m
BTC-USDT 1H
ETH-USDT 30m
ETH-USDT 1H
SOL-USDT 30m
SOL-USDT 1H


In [ ]:
final_df = pd.concat(
    dataset,
    ignore_index=True
)

final_df.dropna(
    inplace=True
)

print("Before balancing:")
print(final_df["TARGET"].value_counts())

# ====================================
# BALANCE TARGETS
# ====================================

target0 = final_df[
    final_df["TARGET"] == 0
]

target1 = final_df[
    final_df["TARGET"] == 1
]

n = min(
    len(target0),
    len(target1)
)

target0 = target0.sample(
    n=n,
    random_state=42
)

target1 = target1.sample(
    n=n,
    random_state=42
)

final_df = pd.concat(
    [target0, target1],
    ignore_index=True
)

final_df = final_df.sample(
    frac=1,
    random_state=42
).reset_index(
    drop=True
)

print("\nAfter balancing:")
print(final_df["TARGET"].value_counts())

print(
    final_df.shape
)

save_path = (
"/content/drive/MyDrive/AI_MANUAL_PROJECT/datasets/manual_signal_dataset.csv"
)

final_df.to_csv(
    save_path,
    index=False
)

print(
    "MANUAL SIGNAL DATASET CREATED"
)

Before balancing:
TARGET
0    101159
1     64498
Name: count, dtype: int64

After balancing:
TARGET
0    64498
1    64498
Name: count, dtype: int64
(128996, 31)
MANUAL SIGNAL DATASET CREATED
